# 任务规划与分解

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import os
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware

# 配置模型
model = ChatOpenAI(
    # 多步骤规划任务建议使用能力较强、支持工具调用的模型
    model="zai-org/GLM-5.2",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1",
)

# 搜索工具
tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

def internet_search(query: str, max_results: int = 5) -> dict:
    """搜索互联网获取最新信息。"""
    return tavily_client.search(query, max_results=max_results)

# 创建 Agent，并显式启用 write_todos
agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    middleware=[TodoListMiddleware()],
    system_prompt="""你是一位专业的技术研究员。
面对复杂研究任务时，你会：
1. 先用 write_todos 制定研究计划
2. 逐步执行每个步骤，及时更新进度
3. 将搜索结果写入文件系统整理
4. 最终输出完整的研究报告
""",
)

# 发起一个需要规划的复杂任务
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "请调研 Agent 开发领域的三大 Harness 框架（Deep Agents、Claude Agent SDK、Codex SDK），对比它们的核心能力差异，写一份简要分析报告。"
    }]
})

print(result["messages"][-1].content)

研究报告已完成并写入 `/root/agent_harness_report.md`。以下是核心结论的简要总结：

---

## 三大 Harness 框架对比要点

### 为什么关注 "Harness"

行业共识已从"模型即产品"转向 **"harness（运行支架）即产品"**——一个可靠的 Agent 不仅需要 LLM，还需要一整套管理工具调用循环、上下文压缩、沙箱执行、权限审批、子 Agent 委派的支撑系统。三大框架正是三种不同哲学的回答。

### 三者定位一览

| 框架 | 范式 | 模型中立性 | 来源哲学 |
|------|------|-----------|---------|
| **Deep Agents**（LangChain） | 通用主义 | ⭐⭐⭐ 完全中立，任意模型 | 自顶向下设计 |
| **Claude Agent SDK**（Anthropic） | 深度优化主义 | ❌ 绑定 Claude | 从 Claude Code 提取 |
| **Codex SDK**（OpenAI） | 成品嵌入主义 | ⚠️ OpenAI 导向 | 从 Codex CLI 提取 |

### 核心差异

1. **模型中立是关键分水岭**：Deep Agents 是唯一真正模型无关的，模型/后端/部署三者独立可选；Claude 和 Codex 都将模型与 harness 深度耦合，换取对特定模型工具调用行为的极致调优。

2. **"设计" vs "提取"**：Deep Agents 从 LangGraph 原生构建，抽象层次最高、可组合性最强（可降级到 LangGraph 自定义图）；Claude/Codex 都是从实战编码产品中"提取"出来的，经过真实生产验证但更贴近原始产品形态。

3. **安全模型各异**：Claude 的 Hooks 是不可绕过的策略层（bypassPermissions 也无效），但沙箱仅覆盖 shell；Codex 提供三档沙箱预设 + 网络代理级域名控制 + 独立可用的 `codex sandbox` 命令；Deep Agents 靠多层（路径验证 + HITL + 远程沙箱）。

4. **编排灵活性光谱**：Deep Agents（组件可重组）→ Claude Agent SDK（中庸）→ C